# 特征筛选

1. 在第一个代码单元配置数据路径、标签、分箱参数、筛选阈值和 OOT 切分；
2. 第二个单元列出样本表里可选的 Y 标签；
3. 运行第三个单元开始筛选；筛选完成后默认对最终候选批量绘图，若填写 `PLOT_FEATURES` 变量清单则只画清单中的变量（可包含未入选变量）。
4. 结果按样本表文件名、变量表文件名和标签写入 `feature_screening_reports/{样本文件名}_{变量表文件名}_{标签名}/`，文件名不带日期；图片按特征名命名，不带编号；每张图底部标注该特征纳入候选的原因（清单中的未入选变量标注用户指定及筛选状态）；`PLOT_RESULTS=False` 可关闭绘图。
5. 时间段超过 `PLOT_MAX_PERIODS`（默认 24）时，每个特征先输出一张全周期总览，再输出 `_p01`、`_p02` 等分页明细；各页共用纵轴和色标。周期较多时两张热力图上下排列，趋势图保留全部周期并自动稀疏日期标签。
6. 线上变量多或内存紧张时，把 `FEATURE_CHUNK_SIZE` 设为 50~200 分块读取和筛选；`N_JOBS` / `PLOT_N_JOBS` 控制分箱和绘图并行进程数（-1 为全部 CPU）。分块模式返回的 `data` 只含主键、标签、时间列及本次绘制的变量列。其余性能参数（`DEDUP_MAX_FEATURES`、`DEDUP_SAMPLE_ROWS`、`PLOT_MAX_FEATURES`）如确需限制，可直接传给 `fs.run_feature_screening`。

OOT 只按 `sample_datetime` 时间切分，默认取最近 30%，切点落在时间点边界上；时间列只有一个取值时会报错。切点只用训练期拟合，OOT 只用于评估。入选需同时满足 IV 阈值、单调、OOT 稳定（方向与 PSI）、缺失箱 IV 占比不超限。同一组合重复运行会覆盖同名结果文件。

In [ ]:
from pathlib import Path

import importlib
import feature_screening as fs

importlib.reload(fs)  # 修改 feature_screening.py 后重跑本单元即可生效

# ---------- 数据与标签 ----------
ROOT_DIR = Path(fs.DEFAULT_ROOT_DIR)  # 固定数据根目录，与 notebook 位置无关
SAMPLE_FILE = "samples.csv"
VARIABLE_FILE = "variable_bank_statement_surplus.csv"
TARGET = "y_fpd15_ovd"  # 可选 y_fpd15_ovd / y_fpd7_ovd / y_mob1_5plus 等
OBS_COL = None           # None 时按 TARGET 自动匹配对应 *_obs
FEATURE_PREFIXES = None  # None 表示除主键、样本表同名列外全部数值列；非数值列自动跳过

# ---------- 分箱参数 ----------
N_BINS = 5          # 最多普通箱数（缺失箱不占额度）
MIN_BIN_PCT = 0.05  # 每个普通箱至少占非缺失样本的比例

# ---------- 筛选阈值 ----------
MIN_IV = 0.02               # 入选 IV 阈值（有 OOT 时取训练与 OOT 的较小值）
MAX_MISSING_IV_SHARE = 0.5  # 缺失箱 IV 占整体 IV 的上限；None 不限制
DEDUP_CORR = 0.8            # 入选变量 WOE 相关去重阈值；None 关闭去重

# ---------- OOT 时间切分 ----------
OOT_START = None  # 指定 OOT 起始日期（如 "2026-10-01"），优先于 OOT_RATIO
OOT_RATIO = 0.3   # 按 sample_datetime 取最近 30% 作 OOT；None 关闭 OOT 评估
MAX_PSI = 0.25    # OOT 与训练集分箱占比的最大 PSI；超限判为不稳定

# ---------- 分析绘图 ----------
PLOT_RESULTS = True    # 是否对去重后的最终候选批量绘图
PLOT_FEATURES = None   # 填写变量名列表（如 ["var_a", "var_b"]）时只画这些变量；None 或 [] 画最终候选
PLOT_MAX_PERIODS = 24  # 每页最多时间段数；超过时输出全周期总览 + 分页明细
TIME_FREQ = "week"     # 时间稳定性粒度：day / week / month

# ---------- 运行性能（一般不用改） ----------
N_JOBS = -1               # 分箱并行进程数，-1 表示全部 CPU；无 fork 平台自动单进程
FEATURE_CHUNK_SIZE = None # 线上变量多/内存紧张时设为 50~200，分块读取和筛选
PLOT_N_JOBS = -1          # 并行绘图进程数，-1 表示全部 CPU

In [ ]:
import pandas as pd

sample_columns = pd.read_csv(ROOT_DIR / SAMPLE_FILE, nrows=0).columns
print("样本表可选 Y 标签：")
print([c for c in sample_columns if c.startswith("y_")])
print("变量表列数：", len(pd.read_csv(ROOT_DIR / VARIABLE_FILE, nrows=0).columns))

In [ ]:
data, summary, bin_detail, candidates = fs.run_feature_screening(
    ROOT_DIR,
    min_iv=MIN_IV,
    n_bins=N_BINS,
    min_bin_pct=MIN_BIN_PCT,
    sample_file=SAMPLE_FILE,
    variable_file=VARIABLE_FILE,
    target=TARGET,
    obs_col=OBS_COL,
    feature_prefixes=FEATURE_PREFIXES,
    oot_start=OOT_START,
    oot_ratio=OOT_RATIO,
    max_psi=MAX_PSI,
    max_missing_iv_share=MAX_MISSING_IV_SHARE,
    dedup_corr=DEDUP_CORR,
    n_jobs=N_JOBS,
    feature_chunk_size=FEATURE_CHUNK_SIZE,
    plot_results=PLOT_RESULTS,
    plot_features=PLOT_FEATURES,
    plot_n_jobs=PLOT_N_JOBS,
    plot_max_periods=PLOT_MAX_PERIODS,
    time_freq=TIME_FREQ,
)

summary.head(30)